## HuggingFace Pool: NumPy Random Tensor

A compact end-to-end example that creates a random NumPy tensor, uploads it to Hugging Face Hub with `HuggingFacePool`, retrieves it back, and validates integrity.

### Prerequisites
- `huggingface_hub` installed
- HF dataset or model repo created
- valid HF token with write/read access

### Credentials
This notebook reads values from:
`laila.args`

Expected keys:
- `HF_REPO_ID` (e.g. `amirzadeh/test`)
- `HF_REPO_TYPE` (`dataset` or `model`, default `dataset`)
- `HF_TOKEN`
- `HF_REVISION` (optional, default `main`)


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/home/ubuntu/")


In [3]:
import numpy as np
import laila
laila.read_args("/home/ubuntu/laila/vault/dev_secrets.toml")


/home/ubuntu/laila/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from laila.pool import HuggingFacePool

required = ["HF_REPO_ID", "HF_REPO_TYPE", "HF_TOKEN", "HF_REVISION"]
missing = [name for name in required if getattr(laila.args, name, None) is None]
if missing:
    raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

hf_pool = HuggingFacePool(
    repo_id=laila.args.HF_REPO_ID,
    repo_type=laila.args.HF_REPO_TYPE,
    revision=laila.args.HF_REVISION,
    token=laila.args.HF_TOKEN,
)


In [5]:
laila.add_pool(hf_pool, pool_nickname="hf_pool")


In [6]:
# Create a random NumPy tensor (array)
np.random.seed(42)
random_tensor = np.random.randn(256, 256).astype(np.float32)
wrapped_tensor = laila.constant(data=random_tensor)

print("Tensor shape:", random_tensor.shape)
print("Tensor dtype:", random_tensor.dtype)
print("Global ID:", wrapped_tensor.global_id)


Tensor shape: (256, 256)
Tensor dtype: float32
Global ID: LAILA:ENTRY:GLOBAL_ID:ad9f437c-0722-438f-85c6-99cfd7574ed0


In [7]:
# Upload to Hugging Face pool
future_memorize = laila.memorize(wrapped_tensor, pool_nickname="hf_pool")
print("Before wait:", future_memorize.status)
future_memorize.wait()
print("After wait:", future_memorize.status)


Before wait: FutureStatus.NOT_STARTED
After wait: FutureStatus.FINISHED


In [8]:
# Download from Hugging Face pool
future_remember = laila.remember(wrapped_tensor.global_id, pool_nickname="hf_pool")
future_remember.wait()
recovered = future_remember.result

assert np.array_equal(recovered.data, random_tensor)
print("Recovery verified: arrays are equal")


Recovery verified: arrays are equal


In [9]:
# Optional cleanup
future_forget = laila.forget(wrapped_tensor.global_id, pool_nickname="hf_pool")
future_forget.wait()
print("Cleanup complete")


Cleanup complete
